In [1]:
import os
from google.colab import files

# 1. Create the folder structure required by the assignment
os.makedirs('/content/part_a', exist_ok=True)
os.makedirs('/content/data', exist_ok=True)

# 2. Upload your CSV files from your computer
print("Please upload q1_heart_disease.csv, q2_customers.csv, and q3_retail_promotions.csv:")
uploaded = files.upload()

# 3. Move the uploaded files into the 'data' folder
for filename in uploaded.keys():
    os.rename(filename, f"/content/data/{filename}")

# 4. Change the working directory to 'part_a'
# This is crucial so that '../data/' points to the correct location
os.chdir('/content/part_a')
print("\nEnvironment setup complete. You are now working inside 'part_a'.")

Please upload q1_heart_disease.csv, q2_customers.csv, and q3_retail_promotions.csv:


Saving q3_retail_promotions.csv to q3_retail_promotions.csv

Environment setup complete. You are now working inside 'part_a'.


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

# 1. Date Feature Engineering
df_retail = pd.read_csv('../data/q3_retail_promotions.csv')
df_retail['transaction_date'] = pd.to_datetime(df_retail['transaction_date'])

df_retail['year'] = df_retail['transaction_date'].dt.year
df_retail['month'] = df_retail['transaction_date'].dt.month
df_retail['day_of_week'] = df_retail['transaction_date'].dt.dayofweek
df_retail['is_month_end'] = (df_retail['transaction_date'].dt.day >= 25).astype(int)

# 2. Temporal Train-Test Split
df_retail = df_retail.sort_values('transaction_date')
X = df_retail.drop(['items_sold', 'transaction_date'], axis=1)
y = df_retail['items_sold']

split = int(0.8 * len(df_retail))
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

In [5]:
# 3. Preprocessing Pipeline
cat_features = ['promotion_type', 'location_type', 'store_size']
num_features = ['competition_density', 'year', 'month', 'day_of_week']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
        ('num', StandardScaler(), num_features)
    ])

# 4. Model Training and Evaluation
models = [
    ("Linear Regression", LinearRegression()),
    ("Random Forest", RandomForestRegressor(random_state=42))
]

for name, model in models:
    pipe = Pipeline([('pre', preprocessor), ('reg', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    print(f"\n{name}:")
    print(f"RMSE:{mean_squared_error(y_test, y_pred, squared=False):.2f}")
    print(f"MAE:{mean_absolute_error(y_test, y_pred):.2f}")

    # Parity Plot
    plt.scatter(y_test, y_pred, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r')
    plt.title(f'Parity Plot: {name}')
    plt.show()


Linear Regression:


TypeError: got an unexpected keyword argument 'squared'